# 10  UKRI research grant signals (an innovation and growth signal)

This notebook adds a fifth signal: **which of our companies have taken part in publicly funded
research and development.** A firm that wins UK Research and Innovation (UKRI) funding, for example
an Innovate UK grant, is investing in innovation, which is a strong growth signal and fits the
technology and fast-growth parts of our dataset especially well.

The data is UKRI's Gateway to Research (GtR), a free open API under the Open Government Licence, no
key needed. Each organisation record carries a name and, for most UK bodies, a full postcode, but
**no company number**. So we match by name and confirm with the full postcode, the same careful
approach used for property and contracts.

## What this produces
One `research_grant` signal per matched company (it records that the company appears as a funded or
partner organisation in GtR). Grant counts, funding values and dates would need a second call per
organisation; that is left as an optional enrichment for later.

## How to run
Run NB05 first so `lloyds.duckdb` exists. This notebook needs no downloaded file and no key: it
pulls organisations straight from the GtR API. Run it top to bottom; it updates the same database.

## 1. Install and import

Same as the other notebooks: makes sure DuckDB and RapidFuzz are available, downloading them if they are not already there.

In [ ]:
import sys, subprocess
for pkg in ["duckdb", "rapidfuzz"]:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

Switches the tools on. RapidFuzz compares company names that are close but not identical.

In [ ]:
import duckdb
import pandas as pd
import requests
import re, time
import xml.etree.ElementTree as ET
from collections import defaultdict
from datetime import datetime, timezone
from rapidfuzz import process, fuzz

print("duckdb", duckdb.__version__, "| pandas", pd.__version__)

## 2. Locate the database and set the dials
Find the `lloyds.duckdb` that NB05 built. `GTR_MAX_PAGES` caps how much of the organisation list we
pull (100 per page); set it to `None` for the whole list (slower, but complete).

This box finds the database and sets one dial. Leave `GTR_MAX_PAGES` as a number for a quick trial, or set it to `None` to pull every organisation (takes several minutes).

In [ ]:
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_DIR = Path("/content/drive/MyDrive/Lloyds")     # same folder as NB05
    DB_PATH = WORK_DIR / "lloyds.duckdb"
else:
    WORK_DIR = Path("..").resolve() / "data" / "processed"
    DB_PATH = WORK_DIR / "lloyds.duckdb"

assert DB_PATH.exists(), f"lloyds.duckdb not found at {DB_PATH}. Run NB05 first."
print("DB:", DB_PATH)

GTR_MAX_PAGES = 300      # 100 orgs/page; set to None for the whole list
USER_AGENT = "LloydsBCB-MSc-project/1.0 (academic; contact via GitHub elyokerr)"
NOW = datetime.now(timezone.utc).isoformat(timespec="seconds")

## 3. Helpers
The same name and postcode cleaners as the other notebooks, so matching uses identical normalisation.

This box recreates the same name and postcode cleaners as before, so a company matches the same way everywhere.

In [ ]:
_SUFFIXES = [
    "LIMITED", "LTD", "PLC", "PUBLIC LIMITED COMPANY", "LLP",
    "LIMITED LIABILITY PARTNERSHIP", "LP", "CIC", "CIO",
    "COMPANY", "CO", "AND", "THE",
]
_SUFFIX_RE = re.compile(r"\b(" + "|".join(_SUFFIXES) + r")\b")

def normalise_name(name):
    if name is None:
        return None
    s = str(name).upper()
    s = re.sub(r"[^A-Z0-9 ]", " ", s)
    s = _SUFFIX_RE.sub(" ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s or None

def normalise_postcode(pc):
    if pc is None:
        return None
    s = re.sub(r"\s+", "", str(pc).upper())
    return s or None

print("helpers ok")

## 4. Load the spine and build the match indexes
We pull the company number, normalised name, and postcode for every company, then build an
exact-name index and first-word blocks so the fuzzy step stays fast.

This box reads the master company list and builds two lookups keyed on the simplified name, each carrying the company's postcode so a name match can be confirmed by location.

In [ ]:
con = duckdb.connect(str(DB_PATH))
spine = con.execute("SELECT company_number, name_norm, postcode FROM companies").df()
print(f"spine: {len(spine):,} companies")

by_name = defaultdict(list)     # name_norm -> [(company_number, postcode), ...]
blocks  = defaultdict(list)     # first word -> [(name_norm, company_number, postcode), ...]
for cn, nm, pc in zip(spine["company_number"], spine["name_norm"], spine["postcode"]):
    if not nm:
        continue
    pcn = normalise_postcode(pc)
    by_name[nm].append((cn, pcn))
    blocks[nm.split(" ")[0]].append((nm, cn, pcn))
print(f"exact-name keys: {len(by_name):,} | first-word blocks: {len(blocks):,}")

## 5. The matching ladder
GtR gives no company number, so every match is by name, confirmed by the full postcode where the
organisation record has one. Same tiers as the property notebook, minus the company-number step.

1. `name_exact_postcode` (0.95) - exact normalised name and the postcode agrees.
2. `name_fuzzy_postcode` (~0.8-0.9) - RapidFuzz match above the cutoff, postcode also agrees.
3. `name_exact_unconfirmed` (0.6) - exact name, but the record has no postcode to confirm with.

This box defines the matching ladder: an exact name with a matching postcode first, then a close fuzzy name with a matching postcode. The postcode check stops us confusing two firms that share a name.

In [ ]:
FUZZY_CUTOFF = 90

def _postcode_match(a, b):
    return bool(a and b and a == b)

def match_org(name, org_postcode=None):
    nm = normalise_name(name)
    if not nm:
        return None, 0.0, "no_match"
    pc = normalise_postcode(org_postcode)

    # 1. exact normalised name, confirmed by postcode where the record gives one
    if nm in by_name:
        cands = by_name[nm]
        confirmed = [c for c, p in cands if _postcode_match(pc, p)]
        if confirmed:
            return confirmed[0], 0.95, "name_exact_postcode"
        if pc:
            return None, 0.0, "name_exact_postcode_mismatch"   # same name, different place
        if len(cands) == 1:
            return cands[0][0], 0.6, "name_exact_unconfirmed"
        return None, 0.0, "name_exact_ambiguous"

    # 2. fuzzy within the first-word block, only when the postcode also agrees
    if pc:
        bucket = blocks.get(nm.split(" ")[0])
        if bucket:
            choices = [b[0] for b in bucket]
            for _, score, idx in process.extract(
                    nm, choices, scorer=fuzz.WRatio, score_cutoff=FUZZY_CUTOFF, limit=5):
                if _postcode_match(pc, bucket[idx][2]):
                    return bucket[idx][1], round(score / 100 * 0.9, 3), "name_fuzzy_postcode"

    return None, 0.0, "no_match"

print(match_org("A COMPANY THAT DOES NOT EXIST ZZZ", "ZZ9 9ZZ"))

## 6. Pull organisations from the Gateway to Research API
Page through the GtR organisation list. Each record gives a name and, for most UK bodies, a full
postcode and region. The response is XML, so we read the fields by their tag name (ignoring the
namespace prefix). Results are cached so re-runs are quick.

This box downloads the list of organisations that appear in UK research funding, straight from the free GtR API. For each it keeps the name, the postcode and the region. The result is saved so re-running is instant.

In [ ]:
GTR_URL = "https://gtr.ukri.org/api/organisation"

def _local(tag):
    return tag.split("}")[-1]        # strip the XML namespace, keep the tag name

def _first_field(org_el, field):
    for ch in org_el.iter():
        if _local(ch.tag) == field and ch.text and ch.text.strip():
            return ch.text.strip()
    return None

def fetch_orgs(max_pages=GTR_MAX_PAGES):
    cache = DB_PATH.parent / "cache_gtr_orgs.parquet"
    if cache.exists():
        print("  using cached GtR organisation pull")
        return pd.read_parquet(cache)

    rows, page = [], 1
    while True:
        r = requests.get(GTR_URL, params={"s": 100, "p": page},
                         headers={"User-Agent": USER_AGENT, "Accept": "application/xml"}, timeout=60)
        if r.status_code in (403, 429):
            print(f"  rate limited at page {page}, stopping (re-run later to continue)")
            break
        r.raise_for_status()
        root = ET.fromstring(r.content)
        orgs = [el for el in root.iter() if _local(el.tag) == "organisation"]
        if not orgs:
            break
        for el in orgs:
            rows.append({
                "org_name": _first_field(el, "name"),
                "postcode": _first_field(el, "postCode"),
                "region":   _first_field(el, "region"),
            })
        page += 1
        if max_pages is not None and page > max_pages:
            break
        time.sleep(0.15)   # be polite

    df = pd.DataFrame(rows, columns=["org_name", "postcode", "region"])
    df = df[df["org_name"].notna()].drop_duplicates()
    df.to_parquet(cache, index=False)
    return df

orgs = fetch_orgs()
print(f"  organisations pulled: {len(orgs):,}")
print(f"  with a postcode: {orgs['postcode'].notna().sum():,}")
orgs.head(3)

## 7. Match organisations to the spine and write research_grant signals
Run each organisation through the name ladder, keep the ones that match a company in our dataset,
and write them into `signals`. We clear any previous GtR rows first so re-runs do not double count.

This box runs every organisation through the name and postcode ladder, keeps the ones that match a company in our list, and saves them into the signals table as research-grant events. It clears old GtR rows first so re-running does not double count.

In [ ]:
results = []
for row in orgs.itertuples(index=False):
    cn, conf, method = match_org(row.org_name, row.postcode)
    if cn:
        results.append((cn, row.org_name, row.region, conf, method))

matched = pd.DataFrame(results, columns=["company_number", "org_name", "region",
                                         "confidence", "method"])
# one signal per matched company; keep the strongest match if an org matched more than once
matched = matched.sort_values("confidence", ascending=False).drop_duplicates("company_number")

matched["signal_type"] = "research_grant"
matched["source"] = "ukri_gtr"
matched["signal_date"] = pd.NaT              # grant dates would need a per-org enrichment call
matched["value"] = pd.NA                     # grant values likewise
matched["detail"] = matched["org_name"].astype(str).str.slice(0, 200)
matched["retrieved_at"] = NOW

print(f"organisations matched to a company in the dataset: {len(matched):,} of {len(orgs):,}")
print("\nby method:")
print(matched["method"].value_counts().to_string())

con.execute("DELETE FROM signals WHERE source = 'ukri_gtr'")
ins = matched[["company_number", "signal_type", "signal_date", "value", "detail",
               "source", "confidence", "retrieved_at"]]
con.register("tmp_sig", ins)
con.execute("""INSERT INTO signals
               SELECT company_number, signal_type, signal_date, value, detail,
                      source, confidence, retrieved_at
               FROM tmp_sig""")
con.unregister("tmp_sig")
print(f"\nwritten to signals: {len(ins):,} rows")

## 8. Summary and a look at the result
How many companies in the dataset have taken part in funded research, and how they split by sector.

This box counts how many companies now carry a research-grant signal and shows the split by sector.

In [ ]:
n_companies = con.execute(
    "SELECT count(DISTINCT company_number) FROM signals WHERE source='ukri_gtr'"
).fetchone()[0]
total = con.execute("SELECT count(*) FROM companies").fetchone()[0]
print(f"companies with a research-grant signal: {n_companies:,} of {total:,} ({n_companies/total:.2%})")

sample = con.execute("""
    SELECT c.sector, count(*) AS companies, round(avg(s.confidence), 2) AS avg_conf
    FROM signals s JOIN companies c ON c.company_number = s.company_number
    WHERE s.source = 'ukri_gtr'
    GROUP BY c.sector
    ORDER BY companies DESC
""").df()
sample

## 9. Visualise the matching and the signals
Four pictures: how the organisations narrow down to a confident match, which method did the matching, the matched companies by sector, and by region.

This box draws four charts: the funnel from organisations down to matches, the matches by method, the matched companies by sector, and by region.

In [ ]:
import matplotlib.pyplot as plt

if len(matched) == 0:
    print("no matches to visualise yet. Check the GtR pull worked and NB05 built the full spine.")
else:
    fig, ax = plt.subplots(2, 2, figsize=(13, 9))

    # A. funnel: from organisations down to a confident match
    funnel = {
        "organisations\npulled": len(orgs),
        "has a\npostcode": int(orgs["postcode"].notna().sum()),
        "matched to\nour dataset": len(matched),
    }
    ax[0, 0].bar(list(funnel.keys()), list(funnel.values()), color="#4477aa")
    ax[0, 0].set_title("From organisations to companies matched")
    for i, v in enumerate(funnel.values()):
        ax[0, 0].text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=9)

    # B. matches by method (each maps to a confidence tier)
    mm = matched["method"].value_counts()
    ax[0, 1].barh(list(mm.index[::-1]), list(mm.values[::-1]), color="#228833")
    ax[0, 1].set_title("Matches by method (confidence tier)")
    for i, v in enumerate(mm.values[::-1]):
        ax[0, 1].text(v, i, f" {v:,}", va="center", fontsize=9)

    # C. matched companies by sector
    bysec = con.execute("""
        SELECT c.sector, count(*) AS n
        FROM signals s JOIN companies c ON c.company_number = s.company_number
        WHERE s.source='ukri_gtr'
        GROUP BY c.sector ORDER BY n DESC
    """).df()
    if len(bysec):
        ax[1, 0].barh(bysec["sector"][::-1], bysec["n"][::-1], color="#ccbb44")
        ax[1, 0].set_title("Matched companies by sector")
        ax[1, 0].tick_params(axis="y", labelsize=8)

    # D. matched companies by region
    reg = matched.assign(region=matched["region"].fillna("UNKNOWN")) \
                 .groupby("region").size().sort_values(ascending=False).head(10)
    if len(reg):
        ax[1, 1].barh(list(reg.index[::-1]), list(reg.values[::-1]), color="#ee6677")
        ax[1, 1].set_title("Matched companies by region (top 10)")
        ax[1, 1].tick_params(axis="y", labelsize=8)

    fig.suptitle("NB10 UKRI grants: matching and the signals produced", fontsize=13)
    fig.tight_layout()
    plt.show()

The funnel shows the organisations narrowing down to the ones we matched. Grants reach a subset of firms, mostly the innovative and research-active ones, so expect the strength here to be in which sector and which companies it flags rather than in sheer numbers.

This box saves and closes the database so the new signals are kept.

In [ ]:
con.close()
print("saved:", DB_PATH)

## Notes and what comes next
- GtR has no company number, so matching is by name confirmed by full postcode. Filter on
  `confidence >= 0.95` to keep only the exact-name, postcode-confirmed matches; the
  `name_exact_unconfirmed` rows (0.6) are the ones to treat with care.
- This writes a presence signal (the company appears in funded research). Grant counts, funding
  values and dates would need one extra call per matched organisation to `/api/organisation/{id}/
  projects`; that is a clean optional enrichment for later, and would add a value and a date to each
  signal.
- The signals table now holds five sources: contract_win, hiring, owns_property, trademark and
  research_grant, all keyed on the 8-char company_number. Each is an independent event source; the
  team's feature/harmonising step joins them by company_number.
- Naming note for the team: the modelling branch uses notebook numbers 10-14, so renumber this one
  before merging to main.